In [0]:
# Step 1: Set up the Silver and audit tables
from pyspark.sql import functions as F
from datetime import datetime, timezone

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema_name = "retail_lakehouse"

silver_table = f"{catalog}.{schema_name}.streaming_orders_silver"
audit_table = f"{catalog}.{schema_name}.streaming_data_quality_audit"

silver_df = spark.table(silver_table)

print("Silver table:", silver_table)
print("Audit table:", audit_table)
print("Rows to check:", silver_df.count())

In [0]:
# Step 2: Calculate the main data quality metrics
checks = [
    (
        "order_id_not_null",
        silver_df.filter(F.col("order_id").isNull()).count() == 0
    ),
    (
        "order_id_unique",
        silver_df.groupBy("order_id").count()
        .filter(F.col("count") > 1).count() == 0
    ),
    (
        "quantity_positive",
        silver_df.filter(
            F.col("quantity").isNull() | (F.col("quantity") <= 0)
        ).count() == 0
    ),
    (
        "unit_price_non_negative",
        silver_df.filter(
            F.col("unit_price").isNull() | (F.col("unit_price") < 0)
        ).count() == 0
    ),
    (
        "order_date_valid",
        silver_df.filter(F.col("order_date").isNull()).count() == 0
    ),
    (
        "product_id_not_null",
        silver_df.filter(F.col("product_id").isNull()).count() == 0
    )
]

for check_name, passed in checks:
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

In [0]:
# Step 3: Store the result of each check
run_timestamp = datetime.now(timezone.utc).replace(tzinfo=None)

audit_rows = [
    (run_timestamp, check_name, "PASS" if passed else "FAIL")
    for check_name, passed in checks
]

audit_df = spark.createDataFrame(
    audit_rows,
    ["run_timestamp", "check_name", "status"]
)

display(audit_df)

In [0]:
# Step 4: Append this run to the Delta audit table
if not spark.catalog.tableExists(audit_table):
    audit_df.write.format("delta").saveAsTable(audit_table)
else:
    audit_df.write.format("delta").mode("append").saveAsTable(audit_table)

print("Quality audit saved.")

In [0]:
# Step 5: Review the latest quality checks
audit_history = spark.table(audit_table)

latest_run = audit_history.agg(
    F.max("run_timestamp").alias("latest_timestamp")
).first()["latest_timestamp"]

latest_results = audit_history.filter(
    F.col("run_timestamp") == F.lit(latest_run)
)

display(latest_results.orderBy("check_name"))

In [0]:
# Step 6: Summarize the latest run
failed_checks = latest_results.filter(F.col("status") == "FAIL").count()
total_checks = latest_results.count()

pipeline_status = "PASS" if failed_checks == 0 else "FAIL"

print("Pipeline status:", pipeline_status)
print("Checks executed:", total_checks)
print("Failed checks:", failed_checks)